In [ ]:
%pylab inline
import scipy as sc
from scipy import optimize
from copy import copy
import eucare as ec

In [ ]:
def kawasaki_sum(v):
    angles = np.abs(np.array([e['in_angle'] for e in v.incoming_iter()]))
    assert len(angles) % 2 == 0
    return np.sum(angles * (-1) ** np.arange(len(angles)))
    
def max_kawasaki_sum(vertices):
    if isinstance(vertices, ec.half.HalfEdgeGraph):
        vertices = [v for v in vertices.vertices if not v.on_border()]
    return np.max([kawasaki_sum(v) for v in vertices])

In [ ]:
render_settings = dict(
    figsize=(7, 7),
    #scale=100,
    render_edges=True,
    render_faces=False,
    render_vertices=False,
    face_inset=0,
    for_cutting = True
)

In [ ]:
#TODO: use incircle center for dual graph generation. will give more symmetric results

def delete_rings_around_face(G, face_to_delete, n_rings):
    current_ring = [face_to_delete] if isinstance(face_to_delete, ec.half.Face) else face_to_delete
    for _ in range(n_rings-1):
        next_ring = set(f2 
                        for f1 in current_ring 
                        for v in f1.vertex_iter()
                        for f2 in v.face_iter()
                        if f2 in G.faces and f2 not in current_ring)
        [G.delete_face(f) for f in current_ring]
        current_ring = next_ring
    [G.delete_face(f) for f in current_ring]
        
def doyle_graph(n=25, rings=21, factor=0.2, deletion_rings=5):
    
    G = ec.prototiles.RegularEuclideanTile(n).make_graph(add_positions=True)[0]
    G = ec.half.EuclideanPositionHEG(other=G)
    
    for i in range(rings):
        G = ec.conway.loft_graph(t=factor)(
            G, 
            delete_on_border=False,
            faces=[f for f in G.faces if f.order() == n]
        )
        
    ec.colorization.congruency_colorize(G)
    
    # get the faces to delete and delete them
    central_ngon = next(iter(f for f in G.faces if f.order() == n))
    central_right_edge = next(iter(e for e in central_ngon.halfedge_iter() 
                                   if e.orig['pos'][1] < 0 and e.dest['pos'][1] > 0))
    
    # add the outer face
    outer_face = ec.half.Face(any_side=G.get_any_border())
    G.faces.add(outer_face)
    for e in G.border_edge_iter():
        e.face = outer_face
    G.check_consistency()
    
    def one_ring_outwards(e):
        return e.rev.nex.nex
    right_edge = central_right_edge
    for _ in range((rings+1)//2):
        right_edge = one_ring_outwards(right_edge)
    face_to_delete = right_edge.face
    initial_ring = [face_to_delete, 
                   right_edge.nex.rev.face,
                   right_edge.pre.rev.face]
    singularity = complex(((right_edge.orig['pos'][0]**2 + right_edge.nex.nex.dest['pos'][0]**2)**0.5)*(1-factor), 0)
    print(singularity)
    
    delete_rings_around_face(G, initial_ring, deletion_rings)
    
    [G.delete_face(f) for f in list(G.faces) if f.order() == n]
    #G = ec.conway.gyro_graph()(G, faces=[f for f in G.faces if f.order() == 4])
    #G = ec.conway.dual_graph()(G)
    G.show(**render_settings)
    # transformations in the complex plane
    ps, vs = G.get_position_view()
    k = ps.copy()
    k = np.array([complex(*ki) for ki in k])
    
    a, b, c, d = 1, 0, 1, (4/5) ** 7.5
    #a, b, c, d = 0, 1, 1, 0
    #k = k / np.max(np.abs(k))
    k = k / (k - singularity)

    #k -= complex(*platonic(3)[0].points[-1])
    #k = k ** 2
    #k = np.exp(k*np.pi/5/5)
    
    k -= np.mean(k)
    k = k / np.max(np.abs(k)) * 3
    k = np.stack([k.real, k.imag], axis=-1)
    
    ps[:] = k
    G.recompute_lengths_and_angles()
    
    return G

G = doyle_graph()
G.show(**render_settings)
G.check_consistency()

In [ ]:

from eucare.classifiers import *
import eucare as eu
from eucare.example_tilesets import *
from eucare.example_graphs import *
from eucare import conway
from copy import deepcopy

def test_graph():
    G = from_tiles(eu.example_tilesets.t_3_3_4_3_4(), rings=3)
    #G = from_tiles(eu.example_tilesets.platonic(6), rings=3)

    plotting_kwargs = {
        'figsize': (10, 10),
        'render_faces': True,
        'render_vertices': False,
        'render_edges': False,
        'face_inset': 0,
    }
    def show():
        global G
        cc = congruency_classifier()

        for f in G.faces:
            f['color_key'] = cc.classify(f)

        G.show(**plotting_kwargs)

    show()


    G = conway.gyro_graph()(G)
    G = conway.dual_graph()(G)
    return G


In [ ]:
def fancy_graph():
    #return doyle_graph()
#     hextile0 = ec.prototiles.RegularEuclideanTile(6, edge_labels=[0]*6)
#     hextile1 = ec.prototiles.RegularEuclideanTile(6, edge_labels=[0]*6)
#     tritile = ec.prototiles.RegularEuclideanTile(3, edge_labels=[0]*3)
#     ec.example_tilesets.align_tiles(hextile0, 0, hextile1, 0)
#     ec.example_tilesets.align_tiles(hextile1, 0, tritile, 0)
#     ec.example_tilesets.align_tiles(tritile, 0, tritile, 0)

     #G = ec.example_graphs.from_tiles([tritile, hextile1], 4)
#     G = ec.example_graphs.from_tiles(ec.example_tilesets.platonic(6), 8)
#     central_face = next(iter(f for f in G.faces if np.linalg.norm(f.midpoint()) < 1e-6))
#     delete_rings_around_face(G, central_face, 4)
#     G = ec.conway.ambo_graph()(G, delete_on_border=True)
    
#     # u2 Graph with nice borders
#     G = ec.example_graphs.from_tiles(ec.example_tilesets.u2_4_6_12__3_4_6_4(), 3, base_tile=-1, vertex_based=False)
#     for e in G.border_edges():
#         if e.rev.face.order() == 4 and e.nex.rev.face.order() == 4:
#             G.execute_edge_instruction(e)
#             G.execute_edge_instruction(e.rev.rev.pre.rev)
#     # optionally, merge triangles
#     G = ec.conway.join_graph()(G, faces=[f for f in G.faces if f.order()==3])
    
    # 3.12.12
#     G = ec.example_graphs.from_tiles(ec.example_tilesets.t_3_12_12(), 1)
#     to_expand_from = [e for e in G.border_edges()
#                       if e in G.halfedges and e.on_border() and e.dest.order() == 3]
#     for e in to_expand_from:
#         G.execute_edge_instruction(e)

    #G = ec.example_graphs.from_tiles(ec.example_tilesets.t_4_6_12(), 3)
            
    #G = ec.example_graphs.rosette(11)
    #G.show(**render_settings)
    
    #G = ec.conway.chamfer_graph(t=1/3)(G, faces=[f for f in G.faces if f.order() == 12 and np.linalg.norm(f.midpoint()) > 0.1])
    
    ## just an n-gon
    
    n = 12
    r = 5
    G = ec.prototiles.RegularEuclideanTile(n).make_graph(add_positions=True)[0]
    G = ec.half.EuclideanPositionHEG(other=G)
    # Idea: first chamfer without border-delete, then loft.
    factor = 2.8
#     G = ec.conway.chamfer_graph(t=1/(factor**1.6))(
#         G, 
#         delete_on_border=False,
#         delete_inner_border=False,
#         faces=[f for f in G.faces if f.order() == n]
#     )
    G = ec.conway.chamfer_graph(t=1/(factor))(
        G, 
        delete_on_border=False,
        delete_inner_border=False,
        faces=[f for f in G.faces if f.order() == n]
    )
    for i in range(r):
        G = ec.conway.loft_graph(t=1/factor)(
            G, 
            delete_on_border=False,
            faces=[f for f in G.faces if f.order() == n]
        )
    
    
    #G = ec.conway.kis_graph()(G)
    #G = ec.conway.dual_graph()(G)
    #ec.colorization.congruency_colorize(G)
    G.show(**render_settings)

    ps, vs = G.get_position_view()

    k = ps.copy()
    k = np.array([complex(*ki) for ki in k])
    
    #k = 1/k
    #k -= complex(*platonic(3)[0].points[-1])
    #k = k ** 2
    #k = np.exp(k*np.pi/5/5)
    
    k -= np.mean(k)
    k = k / np.max(np.abs(k)) * 3
    k = np.stack([k.real, k.imag], axis=-1)
    #k[:, 0] *= 0.5

    ps[:] = k
    G.recompute_lengths_and_angles()
    return G

G = fancy_graph()
G.show(**render_settings)
G.check_consistency()

In [ ]:
def concentric_rings(n, rings, factor):
    r = rings
    G = ec.prototiles.RegularEuclideanTile(n).make_graph(add_positions=True)[0]
    G = ec.half.EuclideanPositionHEG(other=G)
    # Idea: first chamfer without border-delete, then loft.
    G = ec.conway.chamfer_graph(t=1/(factor))(
        G, 
        delete_on_border=False,
        delete_inner_border=False,
        faces=[f for f in G.faces if f.order() == n]
    )
    for i in range(r):
        G = ec.conway.loft_graph(t=1/factor)(
            G, 
            delete_on_border=False,
            faces=[f for f in G.faces if f.order() == n]
        )
    ps, vs = G.get_position_view()
    k = ps.copy()
    k = np.array([complex(*ki) for ki in k])
    k -= np.mean(k)
    k = k / np.max(np.abs(k)) * 3
    k = np.stack([k.real, k.imag], axis=-1)
    ps[:] = k
    G.recompute_lengths_and_angles()
    return G


def archimedian_4_6_12(rings=3, smooth_border=False):
    G = ec.example_graphs.from_tiles(ec.example_tilesets.t_4_6_12(), rings)
    to_join = []
    
    if smooth_border:
        for v in G.vertices:
            if v.on_border() and v.order() == 2:
                to_join.append(v)
        for v in to_join:
            G.join_vertex(v)
        
        
    ps, vs = G.get_position_view()

    k = ps.copy()
    k = np.array([complex(*ki) for ki in k])
    
    #k = 1/k
    #k -= complex(*platonic(3)[0].points[-1])
    #k = k ** 2
    #k = np.exp(k*np.pi/5/5)
    
    k -= np.mean(k)
    k = k / np.max(np.abs(k)) * 3
    k = np.stack([k.real, k.imag], axis=-1)
    ps[:] = k
    
    G.recompute_lengths_and_angles()
    return G

G = archimedian_4_6_12(5)
#G = concentric_rings(12, 4, 3.5)
G.show(**render_settings)
G.check_consistency()

In [ ]:
#G = fancy_graph()

# Step 1: Choose direction for every interior edge.
def random_directed_set(edges):
    if isinstance(edges, ec.half.HalfEdgeGraph):
        edges = edges.halfedges
    directed_edges = set()
    for e in edges:
        if e.rev not in directed_edges:
            directed_edges.add(e)
    return directed_edges

print(len(G.halfedges))
directed_edges = random_directed_set([e for e in G.halfedges 
                                      if not (e.on_border() or e.rev.on_border())])
print(f'n edges: {len(directed_edges)}')

# Step 2: Construct array of all vectors of the directed edges, mapping from edge to index
edge_vectors = np.stack([e.orig['pos'] - e.dest['pos'] for e in directed_edges])

dual_vectors = edge_vectors @ ec.base.rotation_matrix(np.pi/2)
dual_directions = dual_vectors / np.linalg.norm(dual_vectors, axis=1, keepdims=True)
edges_to_ids = {e: i for i, e in enumerate(directed_edges)}
print('dual directions shape:', dual_directions.shape)

# Step 3: Formulate constraints as linear problem Ax = 0
# every constraint is a row in the matrix A. Every interior vertex leads to a constraint. Hence, compute one row for each interior vertex.

interior_vertices = [v for v in G.vertices if not v.on_border()]
print('number of interor vertices=constraints:', len(interior_vertices))

rows = []
n_edges = len(directed_edges)
for v in interior_vertices:
    row = np.zeros(n_edges, dtype=np.float32)
    for e in v.outgoing_iter():
        if e in directed_edges:
            row[edges_to_ids[e]] = -1
        else:
            row[edges_to_ids[e.rev]] = 1
    rows.append(row)
B = np.stack(rows)
A = (B[:, None, :] * dual_directions.T[: None]).reshape(-1, n_edges)
print('A.shape:', A.shape)
U = sc.linalg.null_space(A, rcond=1e-7)
print('U.shape:', U.shape)
assert U.shape[1] > 0, f'G does not have a reciprocal figure!'
# Step 4: Formulate and solve least squares problem to make reciprocal graph as 
# similar as possible to result of conway.dual_graph()(G)

# need map face in primal -> vertex in dual!

# need linear map coords in solution space -> dual edge lenghts
# this is just U @ coords
#coords = np.random.rand(U.shape[1])
#print(((U @ coords)[:, None] * dual_directions).shape)

# need linear map dual edge lengths -> dual edge offsets
# this is just dual_directions

# need linear map (dual edge offsets, position of interior_vertices[0]) -> dual vertex positons
to_process = set(G.faces)
anchor = to_process.pop()
coefficients = {anchor: np.zeros(n_edges, dtype=np.float32)}
border = {anchor}
while border:
    new_border = set()
    for f in border:
        for e in f.halfedge_iter():
            f2 = e.rev.face
            if f2 not in coefficients:
                if e in directed_edges:
                    coefficients[f2] = copy(coefficients[f])
                    coefficients[f2][edges_to_ids[e]] = -1
                elif e.rev in directed_edges:
                    coefficients[f2] = copy(coefficients[f])
                    coefficients[f2][edges_to_ids[e.rev]] = 1
                else:
                    continue
                new_border.add(f2)
    border = new_border
assert set(coefficients.keys()) == set(G.faces)

faces = G.faces
n_faces = len(faces)
print('n_faces:', n_faces)
D2P = np.stack([coefficients[f] for f in faces])
print('D2P.shape:', D2P.shape)

M = (D2P @ U)#[:, None] * dual_directions

M = np.moveaxis(np.dot(D2P, np.moveaxis(U[:, :, None] * dual_directions[:, None], 0, 1)), 1, 2)
print('M.shape', M.shape)

# Add two columns to M, corresponding to the offset of the dual graph
xy_columns = np.zeros((n_faces, 2, 2), dtype=np.float32)
xy_columns[:, 0, 0] = 1
xy_columns[:, 1, 1] = 1
M = np.concatenate([xy_columns, M], axis=-1)

# Get 'ground truth' face centers: for now just com of the faces
face_centers = np.stack([f.midpoint() for f in faces])
print('face_centers.shape', face_centers.shape)

# flatten xy
M = M.reshape(n_faces * 2, -1)
face_centers = face_centers.reshape(n_faces * 2)

# solve the least squares problem
sol = sc.optimize.lsq_linear(M, face_centers)
assert sol['success'], f"{sol['message']}"
sol = sol['x']

#sol[2:] *= 1 + np.random.randn(len(sol) - 2) * 0.3
dual_vertices = M @ sol
dual_vertices = dual_vertices.reshape(-1, 2)

#plt.figure(figsize=(5, 5))
#plt.scatter(dual_vertices[:, 0], dual_vertices[:, 1])
#from eucare import plotting
#plotting.set_equal_aspect()
#plt.show()

# Step 5: make reciprocal figure into face graph
D, (_, _, f_map) = G.copy(return_mappings=True)
f2p = {f_map[f]: dual_vertices[i] for i, f in enumerate(faces)}
D = ec.conway.dual_graph()(D)
for v in D.vertices:
    v['pos'] = f2p[v['pre_conway']]

# Step 6: get shrink-rotate graph and apply mapping
SRG, (_, _, f_map) = G.copy(return_mappings=True)
f2p = {f_map[f]: dual_vertices[i] for i, f in enumerate(faces)}
SRG = ec.conway.twist_rotate_graph()(SRG)

for f in SRG.faces:
    if 'pre_conway' in f.attributes:
        f['color_key'] = f.order() + 10
    else:
        f['color_key'] = 0

twistfaces = list(filter(lambda f: 'twistrotate' in f.attributes, SRG.faces))
for f in twistfaces:
    ps, vs = np.array([[v['pos'], v] for v in f.vertex_iter()]).T
    ps = np.stack(ps)
    
    midpoint = np.mean(ps, axis=0, keepdims=True)
    ps = midpoint + (ps - midpoint) * 2
    
    for v, p in zip(vs, ps):
        v['base_pos'] = p
        
alpha, factor = np.pi/5, 0.5
for f in twistfaces:
    ps, vs = np.array([[v['base_pos'], v] for v in f.vertex_iter()]).T
    ps = np.stack(ps)
    rotation_center = f2p[f['pre_conway']]
    
    ps = rotation_center + (ps - rotation_center) @ ec.base.rotation_matrix(alpha) * factor
    
    for v, p in zip(vs, ps):
        v['pos'] = p
        
SRG.recompute_lengths_and_angles()        
#D.add_graph(G)
D.show(**render_settings)
SRG.show(**render_settings)

# join unneccessary boundary vertices
to_join = []
for v in SRG.vertices:
    if v.on_border() and v.order() == 2:
        to_join.append(v)
for v in to_join:
    SRG.join_vertex(v)
SRG.show(**render_settings)

mks = max_kawasaki_sum(SRG)
print(mks)

In [ ]:
%matplotlib notebook
import ipywidgets as widgets
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection, PolyCollection
from eucare.plotting import set_equal_aspect

def reshrinkrotate(alpha, factor, global_scale=1):
    for f in twistfaces:
        ps, vs = np.array([[v['base_pos'], v] for v in f.vertex_iter()]).T
        ps = np.stack(ps)
        rotation_center = f2p[f['pre_conway']]

        ps = rotation_center + (ps - rotation_center) @ ec.base.rotation_matrix(alpha) * factor
        
        if global_scale != 1:
            ps *= global_scale
            
        for v, p in zip(vs, ps):
            v['pos'] = p
            
def get_segments(edges):
    return np.array([[e.orig['pos'], e.dest['pos']] for  e in edges])

def get_polys(faces):
    return [[v['pos'] for v in f.vertex_iter()] for f in faces]

edges = list(random_directed_set(SRG.halfedges))
faces = list(SRG.faces)

#%timeit SRG.show(**render_settings)
#%timeit reshrinkrotate(np.pi/9, 0.7)
#%timeit get_segments(edges)
#%timeit get_polys(faces)

segments = get_segments(edges)
fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(1, 1, 1)
lc = LineCollection(segments, antialiased=True, color='k', linewidth=1)

pc = PolyCollection(get_polys(faces), antialiased=True, color='k')
pc.set_alpha(0.1)

polys = ax.add_collection(pc)
lines = ax.add_collection(lc)

ax.autoscale()
set_equal_aspect()
plt.draw()

alpha_slider = widgets.FloatSlider(0.166666, min=-1, max=1, step=0.02)
factor_slider = widgets.FloatSlider(0.58, min=0, max=6, step=0.05)

last_reparametrized = False
def update(alpha, factor, folded=False, reparametrized=False, scale_folded=False, show_lines=False, show_polys=True):
    alpha = alpha * np.pi
    global last_reparametrized
    if not last_reparametrized:
        gamma = factor / np.sqrt(factor ** 2 - 2 * factor * np.cos(alpha) + 1)
        beta = np.arccos(np.sin(alpha) / np.sqrt(factor ** 2 - 2 * factor * np.cos(alpha) + 1))
    else:
        gamma = factor
        beta = alpha
        # TODO: sign
        alpha = np.arccos((gamma + np.sin(beta)) / np.sqrt(gamma ** 2 + 2 * gamma * np.sin(beta) + 1))
        factor = gamma / np.sqrt(gamma ** 2 + 2 * gamma * np.sin(beta) + 1)
    
    if reparametrized is not last_reparametrized:
        # adjust slider values
        last_reparametrized = reparametrized
        if reparametrized:
            alpha_slider.value = beta / np.pi
            factor_slider.value = gamma
        else:
            alpha_slider.value = alpha / np.pi
            factor_slider.value = factor
    
    if not folded:
        reshrinkrotate(alpha, factor)
    else:
        factor_folded = gamma / np.sqrt(gamma ** 2 - 2 * gamma * np.sin(beta) + 1)
        alpha_folded = np.sign(alpha) * np.arccos((gamma - np.sin(beta)) / np.sqrt(gamma ** 2 - 2 * gamma * np.sin(beta) + 1))
        reshrinkrotate(alpha_folded, factor_folded, 
                       global_scale=1 if not scale_folded else factor/factor_folded)
    lines.set_segments(get_segments(edges) if show_lines else [])
    polys.set_paths(get_polys(faces) if show_polys else [])
    #ax.draw_artist(lc)
    fig.canvas.draw_idle()
    #print(f'gamma {gamma}, beta {beta * 360 / (2 * np.pi)}')

widgets.interact(
    update,
    alpha=alpha_slider,
    factor=factor_slider,
    
);

In [ ]:
[f.attributes.pop('color_key') for f in SRG.faces if 'color_key' in f.attributes]
render_settings_backlit = copy(render_settings)
render_settings_backlit['render_edges'] = False
render_settings_backlit['render_faces'] = True
SRG.show(**render_settings)

In [ ]:
from eucare.redering import SvgwriteRenderer

plotter = SvgwriteRenderer()
plotter.render_graph('test.svg', SRG, height=15)

In [ ]:
to_join = []
for v in SRG.vertices:
    if v.on_border() and v.order() == 2:
        to_join.append(v)
for v in to_join:
    SRG.join_vertex(v)